In [198]:
library(ade4)
library(ggbiplot)

library(tidyverse)
library(colorRamps)
library(wesanderson)
library(plotly)
library(stringr)
library(entropy)  
library(broom)
library(patchwork)
library(dplyr)

options(repr.plot.width=12, repr.plot.heigh=12)

In [199]:
my_palette <- wes_palette("FantasticFox1", 12, type = "continuous")

# Set default color and fill scales globally
update_geom_defaults("bar", list(fill = my_palette[1], color= "white"))
update_geom_defaults("col", list(fill = my_palette[2], color="white"))

scale_fill_discrete <- function(...) scale_fill_manual(values = my_palette, ...)
scale_color_discrete <- function(...) scale_color_manual(values = my_palette, ...)

In [200]:
ThemeMain<-theme( title =element_text(size=16, face='bold'),
                 axis.text.y = element_blank(), 
                 axis.text.x = element_text(color='black'),
                 axis.ticks.y = element_blank(),
                 axis.title.x = element_text(size=16,color='black',face='bold')
                 )


theme_set(theme_bw())
theme_set(ThemeMain)

In [201]:
df <- read.csv("../data/owid-energy-data.csv")

In [202]:
head(df, 5)

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,⋯,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ASEAN (Ember),2000,,NA,NA,NA,NA,NA,NA,NA,⋯,0,NA,NA,NA,NA,NA,0,NA,0,NA
2,ASEAN (Ember),2001,,NA,NA,NA,NA,NA,NA,NA,⋯,0,NA,NA,NA,NA,NA,0,NA,0,NA
3,ASEAN (Ember),2002,,NA,NA,NA,NA,NA,NA,NA,⋯,0,NA,NA,NA,NA,NA,0,NA,0,NA
4,ASEAN (Ember),2003,,NA,NA,NA,NA,NA,NA,NA,⋯,0,NA,NA,NA,NA,NA,0,NA,0,NA
5,ASEAN (Ember),2004,,NA,NA,NA,NA,NA,NA,NA,⋯,0,NA,NA,NA,NA,NA,0,NA,0,NA


In [203]:
RESOURCES = c("biofuel", "coal", "fossil", "gas", "hydro", "low_carbon", "nuclear", "oil",  "solar", "wind", "low_carbon", "renewables", "other_renewables")

In [204]:
COUNTRY_INFO = c("country", "year", "gdp", "iso_code", "population", "greenhouse_gas_emissions", "carbon_intensity_elec", "net_elec_imports", "net_elec_imports_share_demand", "per_capita_electricity", "electricity_demand", "electricity_generation", "electricity_share_energy", "energy_cons_change_pct", "energy_cons_change_twh", "energy_per_capita", "energy_per_gdp", "primary_energy_consumption")


In [205]:
str(df)

'data.frame':	21812 obs. of  129 variables:
 $ country                                     : chr  "ASEAN (Ember)" "ASEAN (Ember)" "ASEAN (Ember)" "ASEAN (Ember)" ...
 $ year                                        : int  2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 ...
 $ iso_code                                    : chr  "" "" "" "" ...
 $ population                                  : num  NA NA NA NA NA NA NA NA NA NA ...
 $ gdp                                         : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_cons_change_pct                     : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_cons_change_twh                     : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_cons_per_capita                     : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_consumption                         : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_elec_per_capita                     : num  NA NA NA NA NA NA NA NA NA NA ...
 $ biofuel_electricity                  

In [215]:
df_long <- df %>%
  pivot_longer(
    cols =  -all_of(COUNTRY_INFO),
    names_to = "resource_stat",
    values_to = "value"
  ) %>% 
  mutate(
    resource_stat = gsub("fossil_fuel", "fossil",
                    gsub("renewable_", "renewables_", resource_stat))
  )%>%
  mutate(
    resource = str_extract(resource_stat, str_c(RESOURCES, collapse = "|")),
    stat = str_remove(resource_stat, str_c(resource, "_"))
  )%>%
  dplyr::rename(
    country_energy_per_capita = energy_per_capita
  ) %>%
  select(-resource_stat) %>%
  pivot_wider(
    names_from = stat,
    values_from = value
  )

In [216]:
head(df_long)

country,year,iso_code,population,gdp,carbon_intensity_elec,electricity_demand,electricity_generation,electricity_share_energy,energy_cons_change_pct,⋯,share_elec,share_energy,prod_change_pct,prod_change_twh,prod_per_capita,production,energy_per_capita,exc_biofuel_electricity,elec_per_capita_exc_biofuel,share_elec_exc_biofuel
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,1.550,NA,NA,NA,NA,NA,NA,NA,NA,NA
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,20.081,NA,NA,NA,NA,NA,NA,NA,NA,NA
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,80.653,NA,NA,NA,NA,NA,NA,NA,NA,NA
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,43.385,NA,NA,NA,NA,NA,NA,NA,NA,NA
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,13.325,NA,NA,NA,NA,NA,NA,NA,NA,NA
ASEAN (Ember),2000,,NA,NA,569.557,378.61,378.61,NA,NA,⋯,19.347,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [223]:
df_numeric = select(df, where(is.numeric))

In [226]:
df_filled

year,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,biofuel_electricity,biofuel_share_elec,⋯,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2000,NA,NA,NA,NA,NA,NA,NA,5.87,1.550,⋯,0.000,NA,NA,NA,NA,NA,0.00,NA,0.000,NA
2001,NA,NA,NA,NA,NA,NA,NA,6.46,1.596,⋯,0.000,NA,NA,NA,NA,NA,0.00,NA,0.000,NA
2002,NA,NA,NA,NA,NA,NA,NA,6.62,1.528,⋯,0.000,NA,NA,NA,NA,NA,0.00,NA,0.000,NA
2003,NA,NA,NA,NA,NA,NA,NA,7.45,1.626,⋯,0.000,NA,NA,NA,NA,NA,0.00,NA,0.000,NA
2004,NA,NA,NA,NA,NA,NA,NA,8.40,1.692,⋯,0.000,NA,NA,NA,NA,NA,0.00,NA,0.000,NA
2005,NA,NA,NA,NA,NA,NA,NA,8.80,1.684,⋯,0.000,NA,NA,NA,NA,NA,0.02,NA,0.004,NA
2006,NA,NA,NA,NA,NA,NA,NA,8.53,1.559,⋯,0.007,NA,NA,NA,NA,NA,0.05,NA,0.009,NA
2007,NA,NA,NA,NA,NA,NA,NA,10.25,1.770,⋯,0.007,NA,NA,NA,NA,NA,0.06,NA,0.010,NA
2008,NA,NA,NA,NA,NA,NA,NA,10.55,1.747,⋯,0.007,NA,NA,NA,NA,NA,0.06,NA,0.010,NA


In [227]:
pca = prcomp(df_filled, scale. = TRUE)

ERROR: Error in svd(x, nu = 0, nv = k): infinite or missing values in 'x'


In [228]:
ggbiplot(pca, scale = 1)

ERROR: Error in ggbiplot(pca, scale = 1): object 'pca' not found
